# Install Dependencies

In [ ]:
!pip install transformers seqeval

# 1. Split dataset into 70 20 10 + balanced

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from IPython.display import display

# reproducibility
np.random.seed(42)

# load data
df = pd.concat([
    pd.read_csv("train_clean.csv"),
    pd.read_csv("test_clean.csv"),
    pd.read_csv("validation_clean.csv")
], ignore_index=True)

# build document-level label
doc_labels = df.groupby("document_id")["entity_type"].unique().reset_index()

# convert label list → string (for stratification)
doc_labels["label"] = doc_labels["entity_type"].apply(lambda x: "_".join(sorted(x)))

# split doc_ids
train_ids, temp_ids = train_test_split(
    doc_labels["document_id"],
    test_size=0.3,
    stratify=doc_labels["label"],
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=1/3,  # 0.1 total
    stratify=doc_labels.set_index("document_id").loc[temp_ids]["label"],
    random_state=42
)

# build final datasets
train_df = df[df["document_id"].isin(train_ids)]
val_df = df[df["document_id"].isin(val_ids)]
test_df = df[df["document_id"].isin(test_ids)]

# --- PRO TABLE OUTPUT ---
summary_df = pd.DataFrame({
    "Train Size": [f"{len(train_df):,}"],      # Added commas for thousand separators
    "Validation Size": [f"{len(val_df):,}"],
    "Test Size": [f"{len(test_df):,}"]
})

# Apply custom styles (hiding index, adding a title, and styling headers)
pro_table = (
    summary_df.style
    .hide(axis="index")
    .set_caption("Dataset Split Summary")
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'), ('text-align', 'center'), ('padding', '10px')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '10px'), ('font-size', '14px')]}
    ])
)

display(pro_table)

Train Size,Validation Size,Test Size
"20,243","6,171","3,010"


# 3. EDA AND DATA CLEANING

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Initialize datasets from the previous split
train = train_df.copy()
val = val_df.copy()
test = test_df.copy()

# ==========================================
# TABLE 1: Column Info & Missing Values
# ==========================================
display(Markdown("## Column Info & Missing Values (All Splits)"))

info_data = []
for col in train.columns:
    info_data.append({
        "Feature": col,
        "Data Type": str(train[col].dtype),
        "Train Missing": train[col].isnull().sum(),
        "Val Missing": val[col].isnull().sum(),
        "Test Missing": test[col].isnull().sum()
    })

info_df = pd.DataFrame(info_data)
display(info_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]}
]))

# ==========================================
# DATA CLEANING
# ==========================================
display(Markdown("## Validation Data Cleaning"))
val_before = len(val)
val = val.dropna(subset=["entity_text"])
val_after = len(val)

print(f"Dropped {val_before - val_after} rows with missing 'entity_text' from Validation.")
print(f"Validation size after cleaning: {val_after:,}\n")

# ==========================================
# TABLE 2: Entity Type Distribution
# ==========================================
display(Markdown("## Entity Type Distribution"))

dist_df = pd.DataFrame({
    "Entity Type": ["Chemical", "Disease"],
    "Train": [
        f"{(train['entity_type'] == 'Chemical').sum():,}",
        f"{(train['entity_type'] == 'Disease').sum():,}"
    ],
    "Validation": [
        f"{(val['entity_type'] == 'Chemical').sum():,}",
        f"{(val['entity_type'] == 'Disease').sum():,}"
    ],
    "Test": [
        f"{(test['entity_type'] == 'Chemical').sum():,}",
        f"{(test['entity_type'] == 'Disease').sum():,}"
    ]
})

display(dist_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2196F3'), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]}
]))

# ==========================================
# DATA PREVIEW
# ==========================================
display(Markdown("**Data Preview (First 5 Rows & 5 Columns):**"))
preview_cols = ["document_id", "passage_type", "entity_text", "entity_type", "text"]
preview_df = train[preview_cols].head(5).copy()

# Shorten the 'text' column so it doesn't stretch the screen
preview_df["text"] = preview_df["text"].apply(lambda x: x[:75] + "..." if len(str(x)) > 75 else x)

display(preview_df)

## Column Info & Missing Values (All Splits)

Feature,Data Type,Train Missing,Val Missing,Test Missing
document_id,int64,0,0,0
passage_type,object,0,0,0
text,object,0,0,0
entity_text,object,0,7,0
entity_type,object,0,0,0
start,int64,0,0,0
end,int64,0,0,0


## Validation Data Cleaning

Dropped 7 rows with missing 'entity_text' from Validation.
Validation size after cleaning: 6,164



## Entity Type Distribution

Entity Type,Train,Validation,Test
Chemical,"10,969","3,392","1,595"
Disease,"9,274","2,772","1,415"


**Data Preview (First 5 Rows & 5 Columns):**

,document_id,passage_type,entity_text,entity_type,text
0,227508,title,Naloxone,Chemical,Naloxone reverses the antihypertensive effect ...
1,227508,title,clonidine,Chemical,Naloxone reverses the antihypertensive effect ...
2,227508,abstract,hypertensive,Disease,"In unanesthetized, spontaneously hypertensive ..."
3,227508,abstract,clonidine,Chemical,"In unanesthetized, spontaneously hypertensive ..."
4,227508,abstract,nalozone,Chemical,"In unanesthetized, spontaneously hypertensive ..."


# 4. GROUP ENTITIES

In [ ]:
def group_entities_fixed(df):
    grouped = df.groupby("text")
    data = []
    for text, group in grouped:
        entities = []
        for _, row in group.iterrows():
            entity_text = row["entity_text"]
            start = text.find(entity_text)
            if start == -1:
                continue  # skip broken cases
            end = start + len(entity_text)
            entities.append({
                "start": start,
                "end": end,
                "label": row["entity_type"]
            })
        data.append({
            "text": text,
            "entities": entities
        })
    return data

train_grouped = group_entities_fixed(train)
val_grouped = group_entities_fixed(val)
test_grouped = group_entities_fixed(test)

# 5. CONVERT TO BIO

In [ ]:
import re
import pandas as pd
from IPython.display import display, Markdown

def text_to_bio(text, entities):
    tokens = []
    labels = []
    for match in re.finditer(r'\w+|[^\w\s]', text):
        token = match.group()
        start = match.start()
        end = match.end()
        label = "O"
        for ent in entities:
            ent_start = ent["start"]
            ent_end = ent["end"]
            ent_label = ent["label"]
            if not (end <= ent_start or start >= ent_end):
                if re.match(r'\w+', token):
                    if start == ent_start:
                        label = "B-" + ent_label
                    else:
                        label = "I-" + ent_label
        tokens.append(token)
        labels.append(label)
    return tokens, labels

def convert_dataset(grouped_data):
    dataset = []
    for item in grouped_data:
        tokens, labels = text_to_bio(item["text"], item["entities"])
        dataset.append({
            "tokens": tokens,
            "labels": labels
        })
    return dataset

# 1. Convert all datasets
train_bio = convert_dataset(train_grouped)
val_bio = convert_dataset(val_grouped)
test_bio = convert_dataset(test_grouped)

# ==========================================
# SHOW EXAMPLE OF BIO CONVERSION (VECTORS)
# ==========================================
display(Markdown("##BIO Format Example (Vectors)"))

example = train_bio[0]

# Display as raw Python lists (vectors)
print("TOKENS VECTOR:")
print(example["tokens"][:15], "...\n")

print("LABELS VECTOR:")
print(example["labels"][:15], "...\n")

# Display as a horizontal 1xN Table
display(Markdown("**Horizontal Table View (First 15 tokens):**"))

# Create a DataFrame where columns are tokens and the single row is the labels
horizontal_df = pd.DataFrame([example["labels"][:15]], columns=example["tokens"][:15], index=["BIO Tag"])

# FIXED STYLING: Added ('color', 'black') to 'th' so it pops in Dark Mode
display(horizontal_df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('color', 'black'), ('border', '1px solid #ddd'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('text-align', 'center'), ('padding', '8px')]}
]))

##BIO Format Example (Vectors)

TOKENS VECTOR:
['-', 'Tacrolimus', '(', 'FK', '506', ')', 'is', 'a', 'powerful', ',', 'widely', 'used', 'immunosuppressant', '.', 'The'] ...

LABELS VECTOR:
['O', 'B-Chemical', 'O', 'B-Chemical', 'I-Chemical', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...



**Horizontal Table View (First 15 tokens):**

,-,Tacrolimus,(,FK,506,),is,a,powerful,",",widely,used,immunosuppressant,.,The
BIO Tag,O,B-Chemical,O,B-Chemical,I-Chemical,O,O,O,O,O,O,O,O,O,O


# 6. PREPARE FOR BIOLINKBERT & LABEL MAPPING

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

label_list = ["O", "B-Chemical", "I-Chemical", "B-Disease", "I-Disease"]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# ==========================================
# SHOW LABEL DICTIONARY
# ==========================================
display(Markdown("##Label Dictionary Mapping"))

# Convert the dictionary to a DataFrame for clean display
label_df = pd.DataFrame(list(label2id.items()), columns=["Label", "ID ID"])

# Apply styling (Orange header to match the previous dictionary table)
display(label_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#FF9800'), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px'), ('font-family', 'monospace')]}
]))

##Label Dictionary Mapping

Label,ID ID
O,0
B-Chemical,1
I-Chemical,2
B-Disease,3
I-Disease,4


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "michiyasunaga/BioLinkBERT-base"

print(f"Loading Tokenizer & Model: {model_name}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize model with the correct label mappings
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading Tokenizer & Model: michiyasunaga/BioLinkBERT-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: michiyasunaga/BioLinkBERT-base
Key                          | Status     | 
-----------------------------+------------+-
bert.pooler.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids | UNEXPECTED | 
bert.pooler.dense.weight     | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 7. TOKENIZATION + ALIGNMENT

In [ ]:
import re

def convert_to_word_level_bio(item):
    text = item["text"]
    entities = item["entities"]

    # Tokenize text into words, considering punctuation
    words_and_spans = []
    # Regex to find words (alphanumeric sequences) and non-whitespace, non-alphanumeric characters
    # This splits "word." into ["word", "."]
    token_pattern = re.compile(r"\w+|[^\w\s]")

    for match in token_pattern.finditer(text):
        word = match.group(0)
        start = match.start()
        end = match.end()
        words_and_spans.append({"word": word, "start": start, "end": end})

    # Initialize labels for each word token
    word_labels = ["O"] * len(words_and_spans)

    # Align entities to words
    for entity in entities:
        entity_start = entity["start"]
        entity_end = entity["end"]
        entity_label = entity["label"]

        b_tag_assigned = False
        for i, word_info in enumerate(words_and_spans):
            word_start = word_info["start"]
            word_end = word_info["end"]

            # Check for overlap between word and entity
            if max(word_start, entity_start) < min(word_end, entity_end):
                if not b_tag_assigned:
                    word_labels[i] = "B-" + entity_label
                    b_tag_assigned = True
                else:
                    word_labels[i] = "I-" + entity_label
            elif b_tag_assigned and word_start >= entity_end:
                # If we were in an entity and now the word is completely after it, reset flag
                b_tag_assigned = False

    return {
        "tokens": [ws["word"] for ws in words_and_spans],
        "labels": word_labels
    }

def tokenize_and_align_labels(dataset):
    tokenized_data = []
    for item in dataset:
        tokens = item["tokens"]
        labels = item["labels"]

        # BioLinkBERT's max length is typically 512
        tokenized = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=512)
        word_ids = tokenized.word_ids()

        aligned_labels = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(label2id[labels[word_idx]])
            else:
                label = labels[word_idx]
                if label.startswith("B-"):
                    label = label.replace("B-", "I-")
                aligned_labels.append(label2id[label])
            previous_word_idx = word_idx

        tokenized["labels"] = aligned_labels
        tokenized_data.append(tokenized)
    return tokenized_data

print("Tokenizing datasets...")

train_prepared = [convert_to_word_level_bio(item) for item in train_grouped]
val_prepared = [convert_to_word_level_bio(item) for item in val_grouped]
test_prepared = [convert_to_word_level_bio(item) for item in test_grouped]

train_tok = tokenize_and_align_labels(train_prepared)
val_tok = tokenize_and_align_labels(val_prepared)
test_tok = tokenize_and_align_labels(test_prepared)
print("Done.")

Tokenizing datasets...
Done.


# 8. TRAIN BIOLINKBERT MODEL

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import os

os.environ["WANDB_DISABLED"] = "true"

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results_biolinkbert_train",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator
)

print("Starting BioLinkBERT training...")
trainer.train()
print("Training complete.")

Starting BioLinkBERT training...


Epoch,Training Loss,Validation Loss
1,0.096867,0.086585
2,0.068595,0.076630
3,0.056002,0.076667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.


# 9. EVALUATION & METRICS

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# 1. PREDICT ON TEST SET
# ==========================================
test_preds, test_labels_raw, _ = trainer.predict(test_tok)
test_predictions = np.argmax(test_preds, axis=2)

true_labels = []
true_predictions = []

for pred, lab in zip(test_predictions, test_labels_raw):
    temp_preds = []
    temp_labels = []
    for p_, l_ in zip(pred, lab):
        if l_ != -100:  # Ignore special tokens
            temp_preds.append(id2label[p_])
            temp_labels.append(id2label[l_])
    true_predictions.append(temp_preds)
    true_labels.append(temp_labels)

# ==========================================
# 2. PREDICT ON VALIDATION SET
# ==========================================
val_preds, val_labels_raw, _ = trainer.predict(val_tok)
val_predictions = np.argmax(val_preds, axis=2)

true_val_labels = []
true_val_predictions = []

for pred, lab in zip(val_predictions, val_labels_raw):
    temp_preds = []
    temp_labels = []
    for p_, l_ in zip(pred, lab):
        if l_ != -100:  # Ignore special tokens
            temp_preds.append(id2label[p_])
            temp_labels.append(id2label[l_])
    true_val_predictions.append(temp_preds)
    true_val_labels.append(temp_labels)

# ==========================================
# 3. OUTPUT FORMATTING (STYLED TABLES)
# ==========================================
display(Markdown("##BERT Model Evaluation Metrics"))

# Global Metrics Table
global_metrics = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1-Score", "Accuracy"],
    "Test Set": [
        precision_score(true_labels, true_predictions),
        recall_score(true_labels, true_predictions),
        f1_score(true_labels, true_predictions),
        accuracy_score(true_labels, true_predictions)
    ],
    "Validation Set": [
        precision_score(true_val_labels, true_val_predictions),
        recall_score(true_val_labels, true_val_predictions),
        f1_score(true_val_labels, true_val_predictions),
        accuracy_score(true_val_labels, true_val_predictions)
    ]
})

display(Markdown("### Global Overall Metrics"))
display(global_metrics.style.hide(axis="index").format({
    "Test Set": "{:.4f}",
    "Validation Set": "{:.4f}"
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2196F3'), ('color', 'white'), ('text-align', 'center'), ('padding', '10px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '10px')]}
]))

# Detailed Classification Reports
test_dict = classification_report(true_labels, true_predictions, output_dict=True)
val_dict = classification_report(true_val_labels, true_val_predictions, output_dict=True)

df_test = pd.DataFrame(test_dict).T
df_val = pd.DataFrame(val_dict).T

# Function to style our detailed reports consistently
def style_detailed_report(df, title, color):
    display(Markdown(f"### {title}"))
    styled = (df.style
        .format("{:.4f}", subset=["precision", "recall", "f1-score"])
        .format("{:.0f}", subset=["support"], na_rep="-")
        .set_table_styles([
            {'selector': 'th', 'props': [(f'background-color', color), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
            {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled)

style_detailed_report(df_test, "Detailed Classification Report (Test Set)", "#4CAF50")
style_detailed_report(df_val, "Detailed Classification Report (Validation Set)", "#FF9800")

##BERT Model Evaluation Metrics

### Global Overall Metrics

Metric,Test Set,Validation Set
Precision,0.8014,0.8074
Recall,0.7963,0.8091
F1-Score,0.7988,0.8083
Accuracy,0.9760,0.9780


### Detailed Classification Report (Test Set)

,precision,recall,f1-score,support
Chemical,0.8543,0.8416,0.8479,808
Disease,0.7552,0.7560,0.7556,910
micro avg,0.8014,0.7963,0.7988,1718
macro avg,0.8047,0.7988,0.8018,1718
weighted avg,0.8018,0.7963,0.7990,1718


### Detailed Classification Report (Validation Set)

,precision,recall,f1-score,support
Chemical,0.8621,0.8578,0.8599,1582
Disease,0.7578,0.7644,0.7611,1719
micro avg,0.8074,0.8091,0.8083,3301
macro avg,0.8100,0.8111,0.8105,3301
weighted avg,0.8078,0.8091,0.8085,3301


# 10. MANUAL TESTING

In [ ]:
import torch
import pandas as pd
from IPython.display import display, Markdown

def predict_sentence_styled(text):
    model.eval()

    # tokenize sentence
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # dynamically get the device the model is on
    device = next(model.parameters()).device
    encoding = {k: v.to(device) for k, v in encoding.items()}

    # prediction
    with torch.no_grad():
        outputs = model(**encoding)

    predictions = torch.argmax(outputs.logits, dim=2)

    # convert tokens
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
    preds = predictions[0].cpu().numpy()

    # convert prediction IDs to label strings
    labels = [id2label[p] for p in preds]

    # Display the original text
    display(Markdown(f"**Input:** *{text}*"))

    # Build the horizontal wrapped grid (12 tokens per line)
    chunk_size = 12
    grid_rows = []

    for i in range(0, len(tokens), chunk_size):
        t_chunk = tokens[i : i + chunk_size]
        l_chunk = labels[i : i + chunk_size]

        # Pad the chunks with empty strings to keep the table perfectly aligned
        t_chunk += [""] * (chunk_size - len(t_chunk))
        l_chunk += [""] * (chunk_size - len(l_chunk))

        grid_rows.append(t_chunk)
        grid_rows.append(l_chunk)

    grid_df = pd.DataFrame(grid_rows)

    # Custom styler to make the predictions pop visually
    def style_pred_grid(x):
        df_style = pd.DataFrame('', index=x.index, columns=x.columns)
        for r in range(len(x)):
            for c in range(len(x.columns)):
                val = str(x.iloc[r, c])
                if r % 2 == 1:  # This is a Label Row
                    if val.startswith('B-') or val.startswith('I-'):
                        # Highlight entities in a nice purple box
                        df_style.iloc[r, c] = 'background-color: #E1BEE7; color: black; font-weight: bold; text-align: center; border-bottom: 3px solid #9C27B0; padding: 6px;'
                    elif val == "O":
                        df_style.iloc[r, c] = 'color: gray; text-align: center; border-bottom: 3px solid #9C27B0; padding: 6px;'
                else:  # This is a Token Row
                    if val != "":
                        if val in ["[CLS]", "[SEP]", "[PAD]"]:
                            # Highlight special BERT tokens in blue
                            df_style.iloc[r, c] = 'font-weight: bold; color: #1976D2; background-color: #E3F2FD; text-align: center; font-size: 14px; padding: 6px;'
                        else:
                            df_style.iloc[r, c] = 'font-weight: bold; color: black; background-color: #f4f4f4; text-align: center; font-size: 14px; padding: 6px;'
        return df_style

    # Render the styled grid
    styled_grid = (grid_df.style
        .hide(axis="index")
        .hide(axis="columns")
        .apply(style_pred_grid, axis=None)
    )
    display(styled_grid)
    display(Markdown("<br>"))  # Adds a small space between different sentences

# ==========================================
# RUN PREDICTIONS
# ==========================================
display(Markdown("##BERT Model Predictions (Manual Testing)"))

test_sentences = [
    "The patient with chronic kidney disease was treated with Lisinopril and showed improvement.",
    "Ibuprofen may worsen gastric ulcers in patients with peptic ulcer disease.",
    "She was diagnosed with type 2 diabetes and prescribed Metformin and insulin therapy.",
    "Long-term use of corticosteroids like Prednisone can lead to osteoporosis.",
    "The combination of Aspirin and Warfarin increases the risk of bleeding complications."
]

for s in test_sentences:
    predict_sentence_styled(s)

##BERT Model Predictions (Manual Testing)

**Input:** *The patient with chronic kidney disease was treated with Lisinopril and showed improvement.*

[CLS],the,patient,with,chronic,kidney,disease,was,treated,with,lis,##ino
O,O,O,O,B-Disease,I-Disease,I-Disease,O,O,O,B-Chemical,I-Chemical
##pri,##l,and,showed,improvement,.,[SEP],,,,,
I-Chemical,I-Chemical,O,O,O,O,O,,,,,


<br>

**Input:** *Ibuprofen may worsen gastric ulcers in patients with peptic ulcer disease.*

[CLS],ibuprofen,may,worsen,gastric,ulcers,in,patients,with,peptic,ulcer,disease
O,B-Chemical,O,O,B-Disease,I-Disease,O,O,O,B-Disease,I-Disease,I-Disease
.,[SEP],,,,,,,,,,
O,O,,,,,,,,,,


... (6 outputs left out)

<br>

**Input:** *The combination of Aspirin and Warfarin increases the risk of bleeding complications.*

[CLS],the,combination,of,aspirin,and,warfarin,increases,the,risk,of,bleeding
O,O,O,O,B-Chemical,O,B-Chemical,O,O,O,O,B-Disease
complications,.,[SEP],,,,,,,,,
O,O,O,,,,,,,,,


<br>

## Save model


In [ ]:
model.save_pretrained("./ner_model")
tokenizer.save_pretrained("./ner_model")

In [ ]:
import shutil

shutil.make_archive("ner_model", 'zip', "./ner_model")

In [ ]:
import pickle

with open("biolinkbert_model.pkl", "wb") as f:
    pickle.dump({
        "model": model,   # make sure this is BioLinkBERT model
        "tokenizer": tokenizer,
        "id2label": id2label,
        "label2id": label2id
    }, f)

save result

In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
import json
import pandas as pd
import os

MODEL_NAME = "BioLinkBERT"

# =========================
# GLOBAL METRICS
# =========================
precision = precision_score(true_labels, true_predictions)
recall = recall_score(true_labels, true_predictions)
f1 = f1_score(true_labels, true_predictions)
accuracy = accuracy_score(true_labels, true_predictions)

# =========================
# PER-ENTITY METRICS
# =========================
report = classification_report(true_labels, true_predictions, output_dict=True)

chemical = report.get("Chemical", {})
disease = report.get("Disease", {})

results = {
    "model": MODEL_NAME,

    # 🔹 Global
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "accuracy": accuracy,

    # 🔹 Chemical
    "Chemical_precision": chemical.get("precision", 0),
    "Chemical_recall": chemical.get("recall", 0),
    "Chemical_f1": chemical.get("f1-score", 0),

    # 🔹 Disease
    "Disease_precision": disease.get("precision", 0),
    "Disease_recall": disease.get("recall", 0),
    "Disease_f1": disease.get("f1-score", 0),
}

# =========================
# SAVE
# =========================
os.makedirs("results", exist_ok=True)

json_path = f"results/{MODEL_NAME}.json"
csv_path = f"results/{MODEL_NAME}.csv"

with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

pd.DataFrame([results]).to_csv(csv_path, index=False)

print("BioLinkBERT results saved!")
print("JSON:", json_path)
print("CSV:", csv_path)
print(results)